In [33]:
# import kagglehub

# # Define a custom path to download the dataset
# custom_path = r"C:\Users\Tanuj Rajput\OneDrive\Desktop\tanujkaggle\10"

# # Re-download dataset to the custom path
# path = kagglehub.dataset_download("aladdinpersson/flickr8kimagescaptions")

# # Move to custom path (if needed)
# import shutil
# shutil.move(path, custom_path)

# print("Dataset moved to:", custom_path)

In [34]:
import pandas as pd
import numpy as np
import re

In [35]:
captions_dict = {}

with open("flickr8k\\captions.txt", "r") as f:
    for line in f:
        line = line.strip()
        if line == '':
            continue
        img_caption, caption = line.split(',', 1)
        img_name = img_caption.split('#')[0]
        caption = caption.strip()

        if img_name not in captions_dict:
            captions_dict[img_name] = []
        captions_dict[img_name].append(caption)
# print(captions_dict)

In [36]:
def preprocess_caption(caption):
    caption = caption.lower()
    caption = re.sub(r'[^\w\s]', '', caption)
    caption = '<start> ' + caption + ' <end>'
    return caption

for img_name in captions_dict:
    new_captions = []
    for caption in captions_dict[img_name]:
        new_caption = preprocess_caption(caption)
        new_captions.append(new_caption)
    captions_dict[img_name] = new_captions

# print(captions_dict)

In [37]:
from tensorflow.keras.preprocessing.text import Tokenizer

all_captions = []
for caption in captions_dict.values():
    all_captions.extend(caption)

tokenizer = Tokenizer(num_words=10000, oov_token='<unk>')
tokenizer.fit_on_texts(all_captions)

captions_seqs_dict = {}
for img_name, captions in captions_dict.items():
    sequence = tokenizer.texts_to_sequences(captions)
    captions_seqs_dict[img_name] = sequence

# print(captions_seqs_dict)

In [38]:
import os
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tqdm import tqdm

image_folder = "flickr8k/images/"

base_model = InceptionV3(weights='imagenet')
model =  Model(inputs=base_model.input, outputs=base_model.layers[-2].output)

def preprocess_image(img_path):
    img = load_img(img_path, target_size=(299, 299))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    return img

image_features = {}
image_list = list(captions_seqs_dict.keys())

for img_name in tqdm(image_list):
    img_path = os.path.join(image_folder, img_name)
    try:
        img_array = preprocess_image(img_path)
        feature = model.predict(img_array, verbose=0)
        image_features[img_name] = feature.flatten()
    except Exception as e:
        print(f"Error processing image {img_name}: {e}" )

100%|██████████| 8091/8091 [21:01<00:00,  6.41it/s]


In [39]:
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(seq) for captions in captions_seqs_dict.values() for seq in captions)
embedding_dim = 256

inputs1 = Input(shape=(2048,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(embedding_dim, activation='relu')(fe1)

inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(256)(se2)

decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 38)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 38, 256)   │  2,260,480 │ input_layer_12[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 2048)      │          0 │ input_layer_11[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 38, 256)   │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 38)        │          0 │ input_layer_12[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │    524,544 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 256)       │    525,312 │ dropout_3[0][0],  │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256)       │          0 │ dense_3[0][0],    │
│                     │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     65,792 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 8830)      │  2,269,310 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,645,438 (21.54 MB)

 Trainable params: 5,645,438 (21.54 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def data_generator(captions_seqs_dict, image_features, vocab_size, max_length, batch_size):
    while True:
        x1, x2, y = [], [], []
        for img_name, caption_list in captions_seqs_dict.items():
            feature = image_features[img_name]
            for seq in caption_list:
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]

                    x1.append(feature)
                    x2.append(in_seq)
                    y.append(out_seq)

                    if len(x1) == batch_size:
                        yield (np.array(x1, dtype = np.float32), np.array(x2, dtype=np.int32)), np.array(y, dtype=np.float32)
                        x1, x2, y = [], [], []

In [41]:
import tensorflow
batch_size = 64
steps = sum(len(v) for v in captions_seqs_dict.values())

dataset = tensorflow.data.Dataset.from_generator(
    lambda: data_generator(captions_seqs_dict, image_features, vocab_size, max_length, batch_size),
    output_signature=(
        (
            tensorflow.TensorSpec(shape=(None, 2048), dtype=tensorflow.float32),
            tensorflow.TensorSpec(shape=(None, max_length), dtype=tensorflow.int32)
        ),
        tensorflow.TensorSpec(shape=(None, vocab_size), dtype=tensorflow.float32)
    )
)
model.fit(dataset, epochs=20, steps_per_epoch=steps // batch_size, verbose=1)

Epoch 1/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 85s 130ms/step - loss: 5.6363
Epoch 2/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 128ms/step - loss: 4.3693
Epoch 3/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - loss: 3.9464
Epoch 4/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - loss: 3.8397
Epoch 5/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - loss: 3.8439
Epoch 6/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 128ms/step - loss: 3.8760
Epoch 7/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - loss: 3.7921
Epoch 8/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 82s 130ms/step - loss: 3.7281
Epoch 9/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 82s 129ms/step - loss: 3.6687
Epoch 10/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - loss: 3.6604
Epoch 11/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 82s 129ms/step - loss: 3.5653
Epoch 12/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - loss: 3.4733
Epoch 13/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - loss: 3.3913
Epoch 14/20
632/632 ━━━━━━━━━━━━━━━━━━━━ 82s 129ms/step - loss: 3.3916
Epoch 15/20
632

In [42]:
# model.save('image_captioning_model.h5')

In [91]:
# def generate_caption(model, tokenizer, photo, max_length):
#     in_text = '<start>'
#     for _ in range(max_length):
#         sequence = tokenizer.texts_to_sequences([in_text])[0]
#         sequence = pad_sequences([sequence], maxlen=max_length, padding='post')
#         yhat = model.predict([photo, sequence], verbose=0)
#         yhat = np.argmax(yhat[0])
#         word = tokenizer.index_word.get(yhat)

#         if word is None or word == "end":
#             break
#         in_text += ' ' + word

#     final_caption = in_text.replace('<start>', '').strip()
#     return final_caption

def generate_caption_beam_search(model, tokenizer, photo, max_length, beam_index=3):
    start_token = tokenizer.word_index['start']
    end_token = tokenizer.word_index['end']

    start_seq = [start_token]
    sequences = [[start_seq, 0.0]]

    while len(sequences[0][0]) < max_length:
        all_candidates = []
        for seq, score in sequences:
            padded_seq = pad_sequences([seq], maxlen=max_length, padding='post')
            yhat = model.predict([photo, padded_seq], verbose=0)[0]
            top_preds = np.argsort(yhat)[-beam_index:]

            for word in top_preds:
                new_seq = seq + [word]
                new_score = score + np.log(yhat[word] + 1e-10)
                all_candidates.append([new_seq, new_score])

        sequences = sorted(all_candidates, key=lambda tup: tup[1], reverse=True)[:beam_index]

    final_seq = sequences[0][0]

    # Convert to words and stop at 'end'
    final_caption = []
    for idx in final_seq:
        word = tokenizer.index_word.get(idx, '')
        if word == 'end':
            break
        if word not in ['start', '<unk>', '']:
            final_caption.append(word)

    return ' '.join(final_caption)

In [92]:
from matplotlib import pyplot as plt
from PIL import Image
from tensorflow.keras.preprocessing import image as keras_image

# def preprocess_image_for_captioning(image_path):
#     img = tensorflow.keras.preprocessing.image.load_img(image_path, target_size=(299, 299))
#     img = tensorflow.keras.preprocessing.image.img_to_array(img)
#     img = tensorflow.keras.applications.inception_v3.preprocess_input(img)
#     return np.expand_dims(img, axis=0)

def preprocess_image_for_captioning(image_array):
    img = keras_image.array_to_img(image_array)  # Convert NumPy array to PIL Image
    img = img.resize((299, 299))  # Resize image
    img_array = keras_image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0  # Normalize if required by your model
    return img_array

def extract_features(image_path, model):
    img = preprocess_image_for_captioning(image_path)
    features = model.predict(img)
    return features

def show_image_with_caption(image_path, caption):
    image = Image.open(image_path)
    plt.imshow(image)
    plt.axis('off')
    plt.title(caption)
    plt.show()

In [95]:
# image_path = r"C:\Users\Tanuj Rajput\OneDrive\Desktop\tanujkaggle\10\flickr8k\images\278105206_df987b0ca0.jpg"
# image_model = InceptionV3(weights='imagenet')
# image_model = Model(image_model.input, image_model.layers[-2].output)
# features = extract_features(image_path, image_model)

# caption = generate_caption_beam_search(model, tokenizer, features, max_length)
# show_image_with_caption(image_path, caption)

In [94]:
import gradio as gr

def caption_image(image):
    # Preprocess and extract features
    img_array = preprocess_image_for_captioning(image)
    feature = image_model.predict(img_array)
    
    # Generate caption
    caption = generate_caption_beam_search(model, tokenizer, feature, max_length)
    return caption

# Create the Gradio interface
interface = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(type="numpy", label="Upload an image"),
    outputs=gr.Textbox(label="Generated Caption"),
    title="Image Captioning with Deep Learning",
    description="Upload an image and get a descriptive caption using a trained deep learning model.",
)

interface.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 379ms/step
